In [0]:
%sql

CREATE CATALOG IF NOT EXISTS r23053585_catalog;

USE CATALOG r23053585_catalog;

CREATE SCHEMA IF NOT EXISTS pipeline_monitoring;

USE SCHEMA pipeline_monitoring;

In [0]:
%sql

USE CATALOG r23053585_catalog;

USE SCHEMA pipeline_monitoring;

CREATE VOLUME IF NOT EXISTS project_volume;

In [0]:
%sql

SELECT
    current_catalog() AS current_catalog,
    current_schema() AS current_schema;

current_catalog,current_schema
r23053585_catalog,pipeline_monitoring


In [0]:
%sql

DESCRIBE VOLUME project_volume;

name,catalog,database,owner,storage_location,volume_type,comment,securable_type,securable_kind
project_volume,r23053585_catalog,pipeline_monitoring,vishakhasah36@gmail.com,,MANAGED,null,VOLUME,VOLUME_DB_STORAGE


In [0]:
%sql

DESCRIBE VOLUME project_volume;

name,catalog,database,owner,storage_location,volume_type,comment,securable_type,securable_kind
project_volume,r23053585_catalog,pipeline_monitoring,vishakhasah36@gmail.com,,MANAGED,null,VOLUME,VOLUME_DB_STORAGE


In [0]:
%sql

CREATE TABLE IF NOT EXISTS dq_rule_config (
    rule_id STRING,
    rule_name STRING,
    source_table STRING,
    target_column STRING,
    rule_description STRING,
    severity STRING,
    is_active BOOLEAN
)
USING DELTA;

In [0]:
%sql

INSERT INTO dq_rule_config
VALUES
(
    'R001',
    'NULL_PRIMARY_KEY',
    'orders',
    'order_id',
    'Detect records where order_id is NULL',
    'CRITICAL',
    TRUE
),
(
    'R002',
    'DUPLICATE_PRIMARY_KEY',
    'orders',
    'order_id',
    'Detect duplicate order_id values',
    'CRITICAL',
    TRUE
),
(
    'R003',
    'INVALID_AMOUNT',
    'orders',
    'amount',
    'Detect negative or invalid transaction amounts',
    'HIGH',
    TRUE
),
(
    'R004',
    'INVALID_TIMESTAMP',
    'orders',
    'order_timestamp',
    'Detect invalid or missing order timestamps',
    'HIGH',
    TRUE
),
(
    'R005',
    'NULL_CUSTOMER_ID',
    'orders',
    'customer_id',
    'Detect records where customer_id is NULL',
    'HIGH',
    TRUE
),
(
    'R006',
    'REFERENTIAL_INTEGRITY',
    'orders',
    'customer_id',
    'Check whether customer_id exists in the customers table',
    'HIGH',
    TRUE
),
(
    'R007',
    'FRESHNESS_CHECK',
    'orders',
    'order_timestamp',
    'Check whether recent order data has arrived',
    'MEDIUM',
    TRUE
);

num_affected_rows,num_inserted_rows
7,7


In [0]:
%sql

SELECT *
FROM dq_rule_config
ORDER BY rule_id;

rule_id,rule_name,source_table,target_column,rule_description,severity,is_active
R001,NULL_PRIMARY_KEY,orders,order_id,Detect records where order_id is NULL,CRITICAL,true
R001,NULL_PRIMARY_KEY,orders,order_id,Detect records where order_id is NULL,CRITICAL,true
R001,NULL_PRIMARY_KEY,orders,order_id,Detect records where order_id is NULL,CRITICAL,true
R002,DUPLICATE_PRIMARY_KEY,orders,order_id,Detect duplicate order_id values,CRITICAL,true
R002,DUPLICATE_PRIMARY_KEY,orders,order_id,Detect duplicate order_id values,CRITICAL,true
R002,DUPLICATE_PRIMARY_KEY,orders,order_id,Detect duplicate order_id values,CRITICAL,true
R003,INVALID_AMOUNT,orders,amount,Detect negative or invalid transaction amounts,HIGH,true
R003,INVALID_AMOUNT,orders,amount,Detect negative or invalid transaction amounts,HIGH,true
R003,INVALID_AMOUNT,orders,amount,Detect negative or invalid transaction amounts,HIGH,true
R004,INVALID_TIMESTAMP,orders,order_timestamp,Detect invalid or missing order timestamps,HIGH,true


In [0]:
%sql

SELECT COUNT(*) AS total_rules
FROM dq_rule_config;

total_rules
21


In [0]:
%sql

SELECT
    rule_id,
    rule_name,
    source_table,
    target_column,
    severity
FROM dq_rule_config
WHERE is_active = TRUE
ORDER BY rule_id;

rule_id,rule_name,source_table,target_column,severity
R001,NULL_PRIMARY_KEY,orders,order_id,CRITICAL
R001,NULL_PRIMARY_KEY,orders,order_id,CRITICAL
R002,DUPLICATE_PRIMARY_KEY,orders,order_id,CRITICAL
R002,DUPLICATE_PRIMARY_KEY,orders,order_id,CRITICAL
R003,INVALID_AMOUNT,orders,amount,HIGH
R003,INVALID_AMOUNT,orders,amount,HIGH
R004,INVALID_TIMESTAMP,orders,order_timestamp,HIGH
R004,INVALID_TIMESTAMP,orders,order_timestamp,HIGH
R005,NULL_CUSTOMER_ID,orders,customer_id,HIGH
R005,NULL_CUSTOMER_ID,orders,customer_id,HIGH


In [0]:
from pyspark.sql import functions as F
from datetime import datetime, timedelta
import random

# ============================================================
# PROJECT CONFIGURATION
# ============================================================

CATALOG = "r23053585_catalog"
SCHEMA = "pipeline_monitoring"

VOLUME_PATH = (
    f"/Volumes/{CATALOG}/{SCHEMA}/project_volume"
)

RAW_PATH = f"{VOLUME_PATH}/raw"

# Fixed seed ensures reproducible data whenever we rerun this cell
random.seed(23053585)

print("Project:", CATALOG)
print("Raw data location:", RAW_PATH)


# ============================================================
# 1. GENERATE CUSTOMERS DATA
# ============================================================

NUM_CUSTOMERS = 5000

cities = [
    "Bhubaneswar",
    "Kolkata",
    "Delhi",
    "Mumbai",
    "Bengaluru",
    "Pune",
    "Hyderabad",
    "Chennai",
    "Kochi",
    "Patna",
    "Jaipur",
    "Ahmedabad"
]

first_names = [
    "Aarav", "Ananya", "Riya", "Rahul", "Sneha",
    "Aditya", "Priya", "Karan", "Neha", "Vikram",
    "Ishita", "Arjun", "Meera", "Rohan", "Kavya"
]

last_names = [
    "Sharma", "Das", "Singh", "Verma", "Patel",
    "Rao", "Mehta", "Gupta", "Roy", "Nair",
    "Khan", "Mishra", "Jain", "Sahu", "Iyer"
]

customers_data = []

for i in range(1, NUM_CUSTOMERS + 1):
    customer_id = f"C{i:05d}"

    customer_name = (
        random.choice(first_names)
        + " "
        + random.choice(last_names)
    )

    city = random.choice(cities)

    registration_date = (
        datetime(2026, 8, 1)
        + timedelta(days=random.randint(0, 30))
    ).strftime("%Y-%m-%d")

    customers_data.append(
        (
            customer_id,
            customer_name,
            city,
            registration_date
        )
    )

customers_columns = [
    "customer_id",
    "customer_name",
    "city",
    "registration_date"
]

customers_df = spark.createDataFrame(
    customers_data,
    customers_columns
)

print("Customers generated:", customers_df.count())


# ============================================================
# 2. GENERATE PRODUCTS DATA
# ============================================================

NUM_PRODUCTS = 500

product_categories = {
    "Electronics": [
        "Laptop",
        "Smartphone",
        "Monitor",
        "Tablet",
        "Camera",
        "Smartwatch"
    ],
    "Accessories": [
        "Keyboard",
        "Mouse",
        "Headphones",
        "USB Cable",
        "Webcam",
        "Power Bank"
    ],
    "Lifestyle": [
        "Backpack",
        "Desk Lamp",
        "Water Bottle",
        "Office Chair",
        "Fitness Band",
        "Travel Bag"
    ],
    "Home": [
        "Coffee Maker",
        "Mixer Grinder",
        "Air Purifier",
        "Table Fan",
        "Storage Box"
    ]
}

products_data = []

for i in range(1, NUM_PRODUCTS + 1):
    product_id = f"P{i:04d}"

    category = random.choice(list(product_categories.keys()))

    product_name = random.choice(product_categories[category])

    # Add a unique number to make product names distinct
    product_name = f"{product_name} Model {i}"

    price = round(random.uniform(250, 75000), 2)

    products_data.append(
        (
            product_id,
            product_name,
            category,
            str(price)
        )
    )

products_columns = [
    "product_id",
    "product_name",
    "category",
    "price"
]

products_df = spark.createDataFrame(
    products_data,
    products_columns
)

print("Products generated:", products_df.count())


# ============================================================
# 3. GENERATE ORDERS DATA
# ============================================================

NUM_ORDERS = 100000

valid_customer_ids = [
    f"C{i:05d}"
    for i in range(1, NUM_CUSTOMERS + 1)
]

valid_product_ids = [
    f"P{i:04d}"
    for i in range(1, NUM_PRODUCTS + 1)
]

order_statuses = [
    "SUCCESS",
    "SUCCESS",
    "SUCCESS",
    "SUCCESS",
    "FAILED",
    "CANCELLED"
]

start_date = datetime(2026, 8, 20)

orders_data = []

for i in range(1, NUM_ORDERS + 1):

    # Normal order ID
    order_id = f"O{i:07d}"

    customer_id = random.choice(valid_customer_ids)

    product_id = random.choice(valid_product_ids)

    amount = round(random.uniform(250, 75000), 2)

    order_timestamp = (
        start_date
        + timedelta(
            days=random.randint(0, 29),
            hours=random.randint(0, 23),
            minutes=random.randint(0, 59),
            seconds=random.randint(0, 59)
        )
    ).strftime("%Y-%m-%d %H:%M:%S")

    status = random.choice(order_statuses)

    orders_data.append(
        (
            order_id,
            customer_id,
            product_id,
            str(amount),
            order_timestamp,
            status
        )
    )


# ============================================================
# 4. INTRODUCE CONTROLLED DATA-QUALITY DEFECTS
# ============================================================

# Helper function to replace a specific field in a tuple
def replace_field(record, index, new_value):
    record = list(record)
    record[index] = new_value
    return tuple(record)


# ------------------------------------------------------------
# Defect 1: NULL order_id
# ------------------------------------------------------------

for index in [100, 200, 300, 400, 500]:
    orders_data[index] = replace_field(
        orders_data[index],
        0,
        None
    )


# ------------------------------------------------------------
# Defect 2: Duplicate order_id
# ------------------------------------------------------------

for source_index, target_index in [
    (1000, 2000),
    (3000, 4000),
    (5000, 6000),
    (7000, 8000),
    (9000, 10000)
]:
    duplicate_id = orders_data[source_index][0]

    orders_data[target_index] = replace_field(
        orders_data[target_index],
        0,
        duplicate_id
    )


# ------------------------------------------------------------
# Defect 3: NULL customer_id
# ------------------------------------------------------------

for index in range(1100, 1600):
    orders_data[index] = replace_field(
        orders_data[index],
        1,
        None
    )


# ------------------------------------------------------------
# Defect 4: Invalid customer references
# ------------------------------------------------------------

for index in range(2100, 2500):
    orders_data[index] = replace_field(
        orders_data[index],
        1,
        f"C99999_{index}"
    )


# ------------------------------------------------------------
# Defect 5: Negative amounts
# ------------------------------------------------------------

for index in range(3100, 3350):
    orders_data[index] = replace_field(
        orders_data[index],
        3,
        "-500.00"
    )


# ------------------------------------------------------------
# Defect 6: Non-numeric amounts
# ------------------------------------------------------------

for index in range(4100, 4250):
    orders_data[index] = replace_field(
        orders_data[index],
        3,
        "not_available"
    )


# ------------------------------------------------------------
# Defect 7: Invalid timestamps
# ------------------------------------------------------------

for index in range(5100, 5300):
    orders_data[index] = replace_field(
        orders_data[index],
        4,
        "invalid_timestamp"
    )


# ------------------------------------------------------------
# Defect 8: NULL timestamps
# ------------------------------------------------------------

for index in range(6100, 6200):
    orders_data[index] = replace_field(
        orders_data[index],
        4,
        None
    )


# ============================================================
# 5. CREATE SPARK DATAFRAMES
# ============================================================

orders_columns = [
    "order_id",
    "customer_id",
    "product_id",
    "amount",
    "order_timestamp",
    "status"
]

orders_df = spark.createDataFrame(
    orders_data,
    orders_columns
)

print("Orders generated:", orders_df.count())


# ============================================================
# 6. WRITE RAW DATA TO VOLUME AS CSV
# ============================================================

(
    customers_df.write
    .mode("overwrite")
    .option("header", True)
    .csv(f"{RAW_PATH}/customers")
)

(
    products_df.write
    .mode("overwrite")
    .option("header", True)
    .csv(f"{RAW_PATH}/products")
)

(
    orders_df.write
    .mode("overwrite")
    .option("header", True)
    .csv(f"{RAW_PATH}/orders")
)

print("--------------------------------------------------")
print("Raw CSV datasets created successfully.")
print("Customers:", customers_df.count())
print("Products:", products_df.count())
print("Orders:", orders_df.count())
print("--------------------------------------------------")

Project: r23053585_catalog
Raw data location: /Volumes/r23053585_catalog/pipeline_monitoring/project_volume/raw
Customers generated: 5000
Products generated: 500
Orders generated: 100000
--------------------------------------------------
Raw CSV datasets created successfully.
Customers: 5000
Products: 500
Orders: 100000
--------------------------------------------------


In [0]:
display(
    dbutils.fs.ls(
        "/Volumes/r23053585_catalog/pipeline_monitoring/project_volume/raw/"
    )
)

path,name,size,modificationTime
dbfs:/Volumes/r23053585_catalog/pipeline_monitoring/project_volume/raw/customers/,customers/,0,1790270242107
dbfs:/Volumes/r23053585_catalog/pipeline_monitoring/project_volume/raw/orders/,orders/,0,1790270242107
dbfs:/Volumes/r23053585_catalog/pipeline_monitoring/project_volume/raw/products/,products/,0,1790270242107


In [0]:
print("Customers:", customers_df.count())
print("Products:", products_df.count())
print("Orders:", orders_df.count())

Customers: 5000
Products: 500
Orders: 100000


In [0]:
from pyspark.sql import functions as F

display(
    orders_df.filter(
        F.col("order_id").isNull()
        | F.col("customer_id").isNull()
        | (F.col("amount") == "-500.00")
        | (F.col("amount") == "not_available")
        | (F.col("order_timestamp") == "invalid_timestamp")
        | F.col("order_timestamp").isNull()
    )
)

order_id,customer_id,product_id,amount,order_timestamp,status
null,C01106,P0386,57782.21,2026-08-30 22:15:54,CANCELLED
null,C04833,P0441,35401.04,2026-09-12 09:34:24,CANCELLED
null,C04087,P0264,24736.84,2026-09-18 06:46:49,CANCELLED
null,C00877,P0102,42647.5,2026-09-08 14:40:31,SUCCESS
null,C00772,P0006,52618.82,2026-09-07 04:12:31,FAILED
O0001101,null,P0110,60321.85,2026-08-20 20:14:54,CANCELLED
O0001102,null,P0378,66018.3,2026-09-16 08:30:07,SUCCESS
O0001103,null,P0283,11970.12,2026-09-01 18:21:44,CANCELLED
O0001104,null,P0289,16753.11,2026-08-23 02:43:05,SUCCESS
O0001105,null,P0193,25764.13,2026-08-27 20:29:16,CANCELLED


In [0]:
from pyspark.sql import functions as F

print(
    "NULL order_id:",
    orders_df.filter(F.col("order_id").isNull()).count()
)

print(
    "NULL customer_id:",
    orders_df.filter(F.col("customer_id").isNull()).count()
)

print(
    "Negative amounts:",
    orders_df.filter(F.col("amount") == "-500.00").count()
)

print(
    "Invalid amounts:",
    orders_df.filter(F.col("amount") == "not_available").count()
)

print(
    "Invalid timestamps:",
    orders_df.filter(
        F.col("order_timestamp") == "invalid_timestamp"
    ).count()
)

print(
    "NULL timestamps:",
    orders_df.filter(
        F.col("order_timestamp").isNull()
    ).count()
)

NULL order_id: 5
NULL customer_id: 500
Negative amounts: 250
Invalid amounts: 150
Invalid timestamps: 200
NULL timestamps: 100


In [0]:
from pyspark.sql import functions as F

# Project paths
CATALOG = "r23053585_catalog"
SCHEMA = "pipeline_monitoring"

VOLUME_PATH = (
    f"/Volumes/{CATALOG}/{SCHEMA}/project_volume"
)

RAW_PATH = f"{VOLUME_PATH}/raw"

# Read customers CSV files.
# inferSchema=False ensures raw values remain strings.
customers_bronze_df = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("inferSchema", False)
    .load(f"{RAW_PATH}/customers")
)

# Display schema
customers_bronze_df.printSchema()

# Display sample records
display(customers_bronze_df.limit(10))

root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- registration_date: string (nullable = true)



customer_id,customer_name,city,registration_date
C02501,Aditya Patel,Kolkata,2026-08-28
C02502,Rohan Mehta,Kolkata,2026-08-02
C02503,Sneha Iyer,Kolkata,2026-08-12
C02504,Ananya Patel,Jaipur,2026-08-29
C02505,Priya Sharma,Mumbai,2026-08-14
C02506,Meera Das,Kochi,2026-08-31
C02507,Vikram Roy,Bhubaneswar,2026-08-17
C02508,Rohan Das,Hyderabad,2026-08-19
C02509,Ishita Gupta,Patna,2026-08-04
C02510,Riya Patel,Patna,2026-08-05


In [0]:
products_bronze_df = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("inferSchema", False)
    .load(f"{RAW_PATH}/products")
)

products_bronze_df.printSchema()

display(products_bronze_df.limit(10))

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: string (nullable = true)



product_id,product_name,category,price
P0438,Headphones Model 438,Accessories,62847.24
P0439,Headphones Model 439,Accessories,64538.25
P0440,Monitor Model 440,Electronics,14617.26
P0441,Smartwatch Model 441,Electronics,10385.43
P0442,Headphones Model 442,Accessories,50496.13
P0443,Storage Box Model 443,Home,18289.01
P0444,Smartphone Model 444,Electronics,39477.58
P0445,Table Fan Model 445,Home,31402.04
P0446,Power Bank Model 446,Accessories,58459.95
P0447,Air Purifier Model 447,Home,17173.71


In [0]:
orders_bronze_df = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("inferSchema", False)
    .load(f"{RAW_PATH}/orders")
)

orders_bronze_df.printSchema()

display(orders_bronze_df.limit(10))

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- order_timestamp: string (nullable = true)
 |-- status: string (nullable = true)



order_id,customer_id,product_id,amount,order_timestamp,status
O0075001,C04019,P0246,47419.18,2026-09-12 06:55:49,SUCCESS
O0075002,C01500,P0348,53752.2,2026-09-04 01:05:01,SUCCESS
O0075003,C00845,P0346,21533.89,2026-08-29 05:18:47,SUCCESS
O0075004,C03242,P0472,41828.87,2026-09-16 08:01:01,CANCELLED
O0075005,C01325,P0142,17320.95,2026-08-29 19:34:13,SUCCESS
O0075006,C00698,P0458,34584.26,2026-08-31 18:27:52,SUCCESS
O0075007,C03513,P0017,1568.23,2026-09-05 22:48:46,SUCCESS
O0075008,C00507,P0043,24134.05,2026-08-26 12:42:04,SUCCESS
O0075009,C04172,P0220,576.34,2026-08-25 10:13:57,CANCELLED
O0075010,C02893,P0414,7802.66,2026-09-17 02:43:05,CANCELLED


In [0]:
# Save customers as a Bronze Delta table
(
    customers_bronze_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("bronze_customers")
)

# Save products as a Bronze Delta table
(
    products_bronze_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("bronze_products")
)

# Save orders as a Bronze Delta table
(
    orders_bronze_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("bronze_orders")
)

print("Bronze Delta tables created successfully.")

Bronze Delta tables created successfully.


In [0]:
print(
    "Bronze customers:",
    spark.table("bronze_customers").count()
)

print(
    "Bronze products:",
    spark.table("bronze_products").count()
)

print(
    "Bronze orders:",
    spark.table("bronze_orders").count()
)

Bronze customers: 5000
Bronze products: 500
Bronze orders: 100000


In [0]:
for table_name in [
    "bronze_customers",
    "bronze_products",
    "bronze_orders"
]:
    print(f"\nSchema for {table_name}:")
    spark.table(table_name).printSchema()


Schema for bronze_customers:
root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- registration_date: string (nullable = true)


Schema for bronze_products:
root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: string (nullable = true)


Schema for bronze_orders:
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- order_timestamp: string (nullable = true)
 |-- status: string (nullable = true)



In [0]:
display(
    spark.table("bronze_orders")
    .filter(
        F.col("order_id").isNull()
        | F.col("customer_id").isNull()
        | (F.col("amount") == "-500.00")
        | (F.col("amount") == "not_available")
        | (F.col("order_timestamp") == "invalid_timestamp")
        | F.col("order_timestamp").isNull()
    )
)

order_id,customer_id,product_id,amount,order_timestamp,status
null,C01106,P0386,57782.21,2026-08-30 22:15:54,CANCELLED
null,C04833,P0441,35401.04,2026-09-12 09:34:24,CANCELLED
null,C04087,P0264,24736.84,2026-09-18 06:46:49,CANCELLED
null,C00877,P0102,42647.5,2026-09-08 14:40:31,SUCCESS
null,C00772,P0006,52618.82,2026-09-07 04:12:31,FAILED
O0001101,null,P0110,60321.85,2026-08-20 20:14:54,CANCELLED
O0001102,null,P0378,66018.3,2026-09-16 08:30:07,SUCCESS
O0001103,null,P0283,11970.12,2026-09-01 18:21:44,CANCELLED
O0001104,null,P0289,16753.11,2026-08-23 02:43:05,SUCCESS
O0001105,null,P0193,25764.13,2026-08-27 20:29:16,CANCELLED


In [0]:
# ============================================
# STEP 3 — CREATE BRONZE TABLES
# ============================================

# Paths to our raw CSV files
customers_path = "/Volumes/r23053585_catalog/pipeline_monitoring/project_volume/raw/customers"
products_path  = "/Volumes/r23053585_catalog/pipeline_monitoring/project_volume/raw/products"
orders_path    = "/Volumes/r23053585_catalog/pipeline_monitoring/project_volume/raw/orders"


# --------------------------------------------
# 1. READ CUSTOMERS CSV
# --------------------------------------------

customers_bronze_df = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("inferSchema", False)
    .load(customers_path)
)

print("Customers loaded:", customers_bronze_df.count())


# --------------------------------------------
# 2. READ PRODUCTS CSV
# --------------------------------------------

products_bronze_df = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("inferSchema", False)
    .load(products_path)
)

print("Products loaded:", products_bronze_df.count())


# --------------------------------------------
# 3. READ ORDERS CSV
# --------------------------------------------

orders_bronze_df = (
    spark.read
    .format("csv")
    .option("header", True)
    .option("inferSchema", False)
    .load(orders_path)
)

print("Orders loaded:", orders_bronze_df.count())


# --------------------------------------------
# 4. WRITE AS BRONZE DELTA TABLES
# --------------------------------------------

customers_bronze_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("r23053585_catalog.pipeline_monitoring.bronze_customers")

products_bronze_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("r23053585_catalog.pipeline_monitoring.bronze_products")

orders_bronze_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("r23053585_catalog.pipeline_monitoring.bronze_orders")


print("Bronze tables created successfully!")

Customers loaded: 5000
Products loaded: 500
Orders loaded: 100000
Bronze tables created successfully!


In [0]:
%sql

SELECT 'customers' AS table_name, COUNT(*) AS row_count
FROM r23053585_catalog.pipeline_monitoring.bronze_customers

UNION ALL

SELECT 'products', COUNT(*)
FROM r23053585_catalog.pipeline_monitoring.bronze_products

UNION ALL

SELECT 'orders', COUNT(*)
FROM r23053585_catalog.pipeline_monitoring.bronze_orders;

table_name,row_count
customers,5000
products,500
orders,100000


In [0]:
# ============================================
# STEP 4.1 — READ BRONZE TABLES
# ============================================

customers_bronze_df = spark.table(
    "r23053585_catalog.pipeline_monitoring.bronze_customers"
)

products_bronze_df = spark.table(
    "r23053585_catalog.pipeline_monitoring.bronze_products"
)

orders_bronze_df = spark.table(
    "r23053585_catalog.pipeline_monitoring.bronze_orders"
)

print("Bronze tables loaded successfully.")

Bronze tables loaded successfully.


In [0]:
# Check the schema of Bronze orders
orders_bronze_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- order_timestamp: string (nullable = true)
 |-- status: string (nullable = true)



In [0]:

%sql

CREATE OR REPLACE TABLE r23053585_catalog.pipeline_monitoring.silver_orders
USING DELTA
AS

SELECT
    order_id,
    customer_id,
    product_id,
    try_cast(amount AS DOUBLE) AS amount,
    try_cast(order_timestamp AS TIMESTAMP) AS order_timestamp

FROM r23053585_catalog.pipeline_monitoring.bronze_orders;

num_affected_rows,num_inserted_rows


In [0]:


%sql

SELECT
    COUNT(*) AS total_rows,
    COUNT(order_id) AS non_null_order_ids,
    COUNT(*) - COUNT(order_id) AS null_order_ids
FROM r23053585_catalog.pipeline_monitoring.silver_orders;

total_rows,non_null_order_ids,null_order_ids
100000,99995,5


In [0]:
%sql



SELECT
    COUNT(*) AS null_amount_records
FROM r23053585_catalog.pipeline_monitoring.silver_orders
WHERE amount IS NULL;

null_amount_records
150


In [0]:
%sql

SELECT
    COUNT(*) AS null_timestamp_records
FROM r23053585_catalog.pipeline_monitoring.silver_orders
WHERE order_timestamp IS NULL;

null_timestamp_records
300


In [0]:
%sql

SELECT
    order_id,
    COUNT(*) AS occurrence_count
FROM r23053585_catalog.pipeline_monitoring.silver_orders
WHERE order_id IS NOT NULL
GROUP BY order_id
HAVING COUNT(*) > 1
ORDER BY occurrence_count DESC;

order_id,occurrence_count
O0007001,2
O0003001,2
O0001001,2
O0005001,2
O0009001,2


In [0]:

%sql

CREATE OR REPLACE TABLE r23053585_catalog.pipeline_monitoring.silver_orders_dedup
USING DELTA
AS

WITH ranked_orders AS (

    SELECT
        order_id,
        customer_id,
        product_id,
        amount,
        order_timestamp,

        ROW_NUMBER() OVER (
            PARTITION BY order_id
            ORDER BY order_timestamp DESC
        ) AS rn

    FROM r23053585_catalog.pipeline_monitoring.silver_orders
    WHERE order_id IS NOT NULL
)

SELECT
    order_id,
    customer_id,
    product_id,
    amount,
    order_timestamp

FROM ranked_orders
WHERE rn = 1;

num_affected_rows,num_inserted_rows


In [0]:

%sql

SELECT
    COUNT(*) AS deduplicated_count
FROM r23053585_catalog.pipeline_monitoring.silver_orders_dedup;

deduplicated_count
99990


In [0]:
%sql

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT order_id) AS unique_order_ids,
    COUNT(*) - COUNT(DISTINCT order_id) AS duplicate_difference
FROM r23053585_catalog.pipeline_monitoring.silver_orders_dedup;

total_rows,unique_order_ids,duplicate_difference
99990,99990,0


In [0]:
%sql

CREATE OR REPLACE TABLE r23053585_catalog.pipeline_monitoring.silver_orders
USING DELTA
AS

SELECT *
FROM r23053585_catalog.pipeline_monitoring.silver_orders_dedup;

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT order_id) AS unique_order_ids,
    COUNT(*) - COUNT(DISTINCT order_id) AS duplicate_difference
FROM r23053585_catalog.pipeline_monitoring.silver_orders;

total_rows,unique_order_ids,duplicate_difference
99990,99990,0


In [0]:
# Read the Bronze customer table

customers_bronze_df = spark.table(
    "r23053585_catalog.pipeline_monitoring.bronze_customers"
)

print("Bronze customers loaded successfully.")

Bronze customers loaded successfully.


In [0]:
customers_bronze_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- registration_date: string (nullable = true)



In [0]:
%sql

CREATE OR REPLACE TABLE r23053585_catalog.pipeline_monitoring.silver_customers
USING DELTA
AS

SELECT
    customer_id,
    customer_name,
    city,
    try_cast(registration_date AS DATE) AS registration_date

FROM r23053585_catalog.pipeline_monitoring.bronze_customers;

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT
    COUNT(*) AS total_customers,
    COUNT(customer_id) AS non_null_customer_ids,
    COUNT(*) - COUNT(customer_id) AS null_customer_ids
FROM r23053585_catalog.pipeline_monitoring.silver_customers;

total_customers,non_null_customer_ids,null_customer_ids
5000,5000,0


In [0]:
%sql

DESCRIBE r23053585_catalog.pipeline_monitoring.silver_customers;

col_name,data_type,comment
customer_id,string,null
customer_name,string,null
city,string,null
registration_date,date,null


In [0]:
# Read Bronze products

products_bronze_df = spark.table(
    "r23053585_catalog.pipeline_monitoring.bronze_products"
)

# Check the actual columns and data types
products_bronze_df.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: string (nullable = true)



In [0]:
%sql

CREATE OR REPLACE TABLE r23053585_catalog.pipeline_monitoring.silver_products
USING DELTA
AS

SELECT
    product_id,
    product_name,
    category,
    try_cast(price AS DOUBLE) AS price

FROM r23053585_catalog.pipeline_monitoring.bronze_products;

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT
    COUNT(*) AS total_products,
    COUNT(product_id) AS non_null_product_ids,
    COUNT(*) - COUNT(product_id) AS null_product_ids
FROM r23053585_catalog.pipeline_monitoring.silver_products;

total_products,non_null_product_ids,null_product_ids
500,500,0


In [0]:
%sql

DESCRIBE r23053585_catalog.pipeline_monitoring.silver_products;

col_name,data_type,comment
product_id,string,null
product_name,string,null
category,string,null
price,double,null


In [0]:
%sql

SHOW TABLES IN r23053585_catalog.pipeline_monitoring;

database,tableName,isTemporary
pipeline_monitoring,bronze_customers,false
pipeline_monitoring,bronze_orders,false
pipeline_monitoring,bronze_products,false
pipeline_monitoring,dq_rule_config,false
pipeline_monitoring,silver_customers,false
pipeline_monitoring,silver_orders,false
pipeline_monitoring,silver_orders_dedup,false
pipeline_monitoring,silver_products,false


In [0]:
%sql

SELECT
    rule_id,
    rule_name,
    source_table,
    target_column,
    severity,
    is_active
FROM r23053585_catalog.pipeline_monitoring.dq_rule_config
WHERE is_active = TRUE
ORDER BY rule_id;

rule_id,rule_name,source_table,target_column,severity,is_active
R001,NULL_PRIMARY_KEY,orders,order_id,CRITICAL,true
R002,DUPLICATE_PRIMARY_KEY,orders,order_id,CRITICAL,true
R003,INVALID_AMOUNT,orders,amount,HIGH,true
R004,INVALID_TIMESTAMP,orders,order_timestamp,HIGH,true
R005,NULL_CUSTOMER_ID,orders,customer_id,HIGH,true
R006,REFERENTIAL_INTEGRITY,orders,customer_id,HIGH,true
R007,FRESHNESS_CHECK,orders,order_timestamp,MEDIUM,true


In [0]:
%sql

SELECT
    'R001' AS rule_id,
    'NULL_PRIMARY_KEY' AS rule_name,
    COUNT(*) AS rows_checked,
    SUM(
        CASE
            WHEN order_id IS NULL THEN 1
            ELSE 0
        END
    ) AS rows_failed
FROM r23053585_catalog.pipeline_monitoring.bronze_orders;

rule_id,rule_name,rows_checked,rows_failed
R001,NULL_PRIMARY_KEY,100000,5


In [0]:
%sql

SELECT
    'R002' AS rule_id,
    'DUPLICATE_PRIMARY_KEY' AS rule_name,
    COUNT(*) AS rows_checked,
    COUNT(*) - COUNT(DISTINCT order_id) AS rows_failed
FROM r23053585_catalog.pipeline_monitoring.bronze_orders
WHERE order_id IS NOT NULL;

rule_id,rule_name,rows_checked,rows_failed
R002,DUPLICATE_PRIMARY_KEY,99995,5


In [0]:
%sql

SELECT
    'R003' AS rule_id,
    'INVALID_AMOUNT' AS rule_name,
    COUNT(*) AS rows_checked,
    SUM(
        CASE
            WHEN try_cast(amount AS DOUBLE) IS NULL
                 OR try_cast(amount AS DOUBLE) < 0
            THEN 1
            ELSE 0
        END
    ) AS rows_failed
FROM r23053585_catalog.pipeline_monitoring.bronze_orders;

rule_id,rule_name,rows_checked,rows_failed
R003,INVALID_AMOUNT,100000,400


In [0]:
%sql

SELECT
    'R004' AS rule_id,
    'INVALID_TIMESTAMP' AS rule_name,
    COUNT(*) AS rows_checked,
    SUM(
        CASE
            WHEN try_cast(order_timestamp AS TIMESTAMP) IS NULL
            THEN 1
            ELSE 0
        END
    ) AS rows_failed
FROM r23053585_catalog.pipeline_monitoring.bronze_orders;

rule_id,rule_name,rows_checked,rows_failed
R004,INVALID_TIMESTAMP,100000,300


In [0]:
%sql

SELECT
    'R005' AS rule_id,
    'NULL_CUSTOMER_ID' AS rule_name,
    COUNT(*) AS rows_checked,
    SUM(
        CASE
            WHEN customer_id IS NULL THEN 1
            ELSE 0
        END
    ) AS rows_failed
FROM r23053585_catalog.pipeline_monitoring.bronze_orders;

rule_id,rule_name,rows_checked,rows_failed
R005,NULL_CUSTOMER_ID,100000,500


In [0]:
%sql

SELECT
    'R006' AS rule_id,
    'REFERENTIAL_INTEGRITY' AS rule_name,
    COUNT(*) AS rows_checked,
    SUM(
        CASE
            WHEN o.customer_id IS NOT NULL
                 AND c.customer_id IS NULL
            THEN 1
            ELSE 0
        END
    ) AS rows_failed

FROM r23053585_catalog.pipeline_monitoring.bronze_orders o

LEFT JOIN
    r23053585_catalog.pipeline_monitoring.bronze_customers c

ON o.customer_id = c.customer_id;

rule_id,rule_name,rows_checked,rows_failed
R006,REFERENTIAL_INTEGRITY,100000,400


In [0]:
%sql

SELECT
    'R007' AS rule_id,
    'FRESHNESS_CHECK' AS rule_name,

    MAX(
        try_cast(order_timestamp AS TIMESTAMP)
    ) AS latest_order_timestamp,

    current_timestamp() AS check_timestamp,

    ROUND(
        (
            unix_timestamp(current_timestamp())
            -
            unix_timestamp(
                MAX(try_cast(order_timestamp AS TIMESTAMP))
            )
        ) / 3600.0,
        2
    ) AS freshness_hours

FROM r23053585_catalog.pipeline_monitoring.bronze_orders;

rule_id,rule_name,latest_order_timestamp,check_timestamp,freshness_hours
R007,FRESHNESS_CHECK,2026-09-18T23:59:56.000Z,2026-09-25T14:37:28.646Z,158.63


In [0]:
%sql
CREATE TABLE IF NOT EXISTS r23053585_catalog.pipeline_monitoring.gold_dq_results (
    table_name STRING,
    load_id STRING,
    rule_id STRING,
    rule_name STRING,
    rule_scope STRING,
    rows_checked BIGINT,
    rows_failed BIGINT,
    fail_rate DOUBLE,
    rule_status STRING,
    check_timestamp TIMESTAMP
)
USING DELTA;

In [0]:
%sql
TRUNCATE TABLE r23053585_catalog.pipeline_monitoring.gold_dq_results;
INSERT INTO r23053585_catalog.pipeline_monitoring.gold_dq_results
SELECT
    'orders' AS table_name,
    'initial_load' AS load_id,
    'R001' AS rule_id,
    'NULL_PRIMARY_KEY' AS rule_name,
    'row' AS rule_scope,
    100000 AS rows_checked,
    5 AS rows_failed,
    5.0 / 100000 AS fail_rate,
    'fail' AS rule_status,
    current_timestamp() AS check_timestamp

UNION ALL

SELECT
    'orders',
    'initial_load',
    'R002',
    'DUPLICATE_PRIMARY_KEY',
    'row',
    99995,
    5,
    5.0 / 99995,
    'fail',
    current_timestamp()

UNION ALL

SELECT
    'orders',
    'initial_load',
    'R003',
    'INVALID_AMOUNT',
    'row',
    100000,
    400,
    400.0 / 100000,
    'fail',
    current_timestamp()

UNION ALL

SELECT
    'orders',
    'initial_load',
    'R004',
    'INVALID_TIMESTAMP',
    'row',
    100000,
    300,
    300.0 / 100000,
    'fail',
    current_timestamp()

UNION ALL

SELECT
    'orders',
    'initial_load',
    'R005',
    'NULL_CUSTOMER_ID',
    'row',
    100000,
    500,
    500.0 / 100000,
    'fail',
    current_timestamp()

UNION ALL

SELECT
    'orders',
    'initial_load',
    'R006',
    'REFERENTIAL_INTEGRITY',
    'row',
    100000,
    400,
    400.0 / 100000,
    'fail',
    current_timestamp()

UNION ALL

SELECT
    'orders',
    'initial_load',
    'R007',
    'FRESHNESS_CHECK',
    'table',
    1,
    0,
    0.0,
    'pass',
    current_timestamp();

num_affected_rows,num_inserted_rows
7,7


In [0]:
%sql
SELECT
    rule_id,
    rule_name,
    rows_checked,
    rows_failed,
    ROUND(fail_rate * 100, 4) AS fail_rate_percent,
    rule_status
FROM r23053585_catalog.pipeline_monitoring.gold_dq_results
ORDER BY rule_id;

rule_id,rule_name,rows_checked,rows_failed,fail_rate_percent,rule_status
R001,NULL_PRIMARY_KEY,100000,5,0.005,fail
R002,DUPLICATE_PRIMARY_KEY,99995,5,0.005,fail
R003,INVALID_AMOUNT,100000,400,0.4,fail
R004,INVALID_TIMESTAMP,100000,300,0.3,fail
R005,NULL_CUSTOMER_ID,100000,500,0.5,fail
R006,REFERENTIAL_INTEGRITY,100000,400,0.4,fail
R007,FRESHNESS_CHECK,1,0,0.0,pass


In [0]:
%sql
CREATE TABLE IF NOT EXISTS r23053585_catalog.pipeline_monitoring.gold_freshness (
    table_name STRING,
    check_ts TIMESTAMP,
    last_business_ts TIMESTAMP,
    hours_behind DOUBLE,
    expected_interval_hours DOUBLE,
    freshness_status STRING
)
USING DELTA;

In [0]:
%sql
TRUNCATE TABLE r23053585_catalog.pipeline_monitoring.gold_freshness;
INSERT INTO r23053585_catalog.pipeline_monitoring.gold_freshness
SELECT
    'orders' AS table_name,
    current_timestamp() AS check_ts,
    MAX(order_timestamp) AS last_business_ts,
    ROUND(
        (
            unix_timestamp(current_timestamp())
            - unix_timestamp(MAX(order_timestamp))
        ) / 3600.0,
        2
    ) AS hours_behind,
    24.0 AS expected_interval_hours,
    CASE
        WHEN (
            (
                unix_timestamp(current_timestamp())
                - unix_timestamp(MAX(order_timestamp))
            ) / 3600.0
        ) <= 24
        THEN 'pass'
        ELSE 'fail'
    END AS freshness_status
FROM r23053585_catalog.pipeline_monitoring.silver_orders;

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
SELECT *
FROM r23053585_catalog.pipeline_monitoring.gold_freshness;

table_name,check_ts,last_business_ts,hours_behind,expected_interval_hours,freshness_status
orders,2026-09-25T16:54:11.143Z,2026-09-18T23:59:56.000Z,160.9,24.0,fail


In [0]:
%sql
SELECT
    table_name,
    rule_id,
    rule_name,
    rule_scope,
    rows_checked,
    rows_failed,
    ROUND(fail_rate * 100, 4) AS fail_rate_percent,
    rule_status
FROM r23053585_catalog.pipeline_monitoring.gold_dq_results
ORDER BY rule_id;

table_name,rule_id,rule_name,rule_scope,rows_checked,rows_failed,fail_rate_percent,rule_status
orders,R001,NULL_PRIMARY_KEY,row,100000,5,0.005,fail
orders,R002,DUPLICATE_PRIMARY_KEY,row,99995,5,0.005,fail
orders,R003,INVALID_AMOUNT,row,100000,400,0.4,fail
orders,R004,INVALID_TIMESTAMP,row,100000,300,0.3,fail
orders,R005,NULL_CUSTOMER_ID,row,100000,500,0.5,fail
orders,R006,REFERENTIAL_INTEGRITY,row,100000,400,0.4,fail
orders,R007,FRESHNESS_CHECK,table,1,0,0.0,pass


In [0]:
%sql
SELECT
    table_name,
    last_business_ts,
    ROUND(hours_behind, 2) AS hours_behind,
    expected_interval_hours,
    freshness_status
FROM r23053585_catalog.pipeline_monitoring.gold_freshness;

table_name,last_business_ts,hours_behind,expected_interval_hours,freshness_status
orders,2026-09-18T23:59:56.000Z,160.9,24.0,fail


In [0]:
gold_dq = spark.table(
    "r23053585_catalog.pipeline_monitoring.gold_dq_results"
)

gold_dq.coalesce(1).write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv(
        "/Volumes/r23053585_catalog/pipeline_monitoring/project_volume/gold_export/dq_results"
    )

In [0]:
display(
    dbutils.fs.ls(
        "/Volumes/r23053585_catalog/pipeline_monitoring/project_volume/gold_export/dq_results"
    )
)

path,name,size,modificationTime
dbfs:/Volumes/r23053585_catalog/pipeline_monitoring/project_volume/gold_export/dq_results/_SUCCESS,_SUCCESS,0,1790359746000
dbfs:/Volumes/r23053585_catalog/pipeline_monitoring/project_volume/gold_export/dq_results/_committed_2352943740251925820,_committed_2352943740251925820,113,1790359745000
dbfs:/Volumes/r23053585_catalog/pipeline_monitoring/project_volume/gold_export/dq_results/_started_2352943740251925820,_started_2352943740251925820,0,1790359745000
dbfs:/Volumes/r23053585_catalog/pipeline_monitoring/project_volume/gold_export/dq_results/part-00000-tid-2352943740251925820-be6cb3e7-304f-4aa1-bfd7-1449ea6038b7-146-1-c000.csv,part-00000-tid-2352943740251925820-be6cb3e7-304f-4aa1-bfd7-1449ea6038b7-146-1-c000.csv,760,1790359745000


In [0]:
gold_freshness = spark.table(
    "r23053585_catalog.pipeline_monitoring.gold_freshness"
)

gold_freshness.coalesce(1).write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv(
        "/Volumes/r23053585_catalog/pipeline_monitoring/project_volume/gold_export/freshness"
    )

In [0]:
display(
    dbutils.fs.ls(
        "/Volumes/r23053585_catalog/pipeline_monitoring/project_volume/gold_export/freshness"
    )
)

path,name,size,modificationTime
dbfs:/Volumes/r23053585_catalog/pipeline_monitoring/project_volume/gold_export/freshness/_SUCCESS,_SUCCESS,0,1790360099000
dbfs:/Volumes/r23053585_catalog/pipeline_monitoring/project_volume/gold_export/freshness/_committed_7996046933608554371,_committed_7996046933608554371,113,1790360099000
dbfs:/Volumes/r23053585_catalog/pipeline_monitoring/project_volume/gold_export/freshness/_started_7996046933608554371,_started_7996046933608554371,0,1790360098000
dbfs:/Volumes/r23053585_catalog/pipeline_monitoring/project_volume/gold_export/freshness/part-00000-tid-7996046933608554371-f2469ec6-6c34-403f-a9b3-5911df85a528-151-1-c000.csv,part-00000-tid-7996046933608554371-f2469ec6-6c34-403f-a9b3-5911df85a528-151-1-c000.csv,164,1790360098000


In [0]:
%sql
SELECT
    'gold_dq_results' AS table_name,
    COUNT(*) AS row_count
FROM r23053585_catalog.pipeline_monitoring.gold_dq_results

UNION ALL

SELECT
    'gold_freshness',
    COUNT(*)
FROM r23053585_catalog.pipeline_monitoring.gold_freshness;

table_name,row_count
gold_dq_results,7
gold_freshness,1


In [0]:
%sql
SELECT
    'gold_dq_results' AS table_name,
    COUNT(*) AS row_count
FROM r23053585_catalog.pipeline_monitoring.gold_dq_results

UNION ALL

SELECT
    'gold_freshness',
    COUNT(*)
FROM r23053585_catalog.pipeline_monitoring.gold_freshness;

table_name,row_count
gold_dq_results,7
gold_freshness,1
